In [37]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [38]:
import os 
print(os.listdir('.'))

['requirements.txt', '.gitignore', '.venv', '.git', 'data', 'hydroponics.ipynb']


In [39]:
file_path = "data/IoTData --Raw--.csv"
if os.path.exists(file_path):
  df = pd.read_csv(file_path)
  print(df.head())
  print(df.info())

   id            timestamp   pH    TDS  water_level  DHT_temp  DHT_humidity  \
0   1  2023-11-26 10:57:52  7.0  500.0            0      25.5          60.0   
1   2  2023-11-26 10:58:37  7.0  500.0            0      25.5          60.0   
2   3  2023-11-26 11:01:34  7.0  500.0            3      25.5          60.0   
3   4  2023-12-05 11:30:58  7.0  500.0            3      25.5          60.0   
4   5  2023-12-05 11:33:50  7.0  500.0            3      25.5          60.0   

   water_temp pH_reducer add_water nutrients_adder humidifier ex_fan  
0        20.0         ON       NaN             OFF        OFF     ON  
1        20.0         ON       NaN             OFF        OFF    OFF  
2        20.0         ON       NaN             OFF        OFF    OFF  
3        20.0         ON       NaN             OFF        OFF    OFF  
4        20.0         ON        ON             OFF        OFF    OFF  
<class 'pandas.DataFrame'>
RangeIndex: 50570 entries, 0 to 50569
Data columns (total 13 columns):
 

In [40]:
original_dataset = pd.read_csv("data/IoTData --Raw--.csv")
df = original_dataset.copy()

In [41]:
df.head()

,id,timestamp,pH,TDS,water_level,DHT_temp,DHT_humidity,water_temp,pH_reducer,add_water,nutrients_adder,humidifier,ex_fan
0,1,2023-11-26 10:57:52,7.0,500.0,0,25.5,60.0,20.0,ON,NaN,OFF,OFF,ON
1,2,2023-11-26 10:58:37,7.0,500.0,0,25.5,60.0,20.0,ON,NaN,OFF,OFF,OFF
2,3,2023-11-26 11:01:34,7.0,500.0,3,25.5,60.0,20.0,ON,NaN,OFF,OFF,OFF
3,4,2023-12-05 11:30:58,7.0,500.0,3,25.5,60.0,20.0,ON,NaN,OFF,OFF,OFF
4,5,2023-12-05 11:33:50,7.0,500.0,3,25.5,60.0,20.0,ON,ON,OFF,OFF,OFF


In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50570 entries, 0 to 50569
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               50570 non-null  int64  
 1   timestamp        50570 non-null  str    
 2   pH               50570 non-null  float64
 3   TDS              50570 non-null  float64
 4   water_level      50570 non-null  int64  
 5   DHT_temp         50570 non-null  float64
 6   DHT_humidity     50570 non-null  float64
 7   water_temp       50570 non-null  float64
 8   pH_reducer       50570 non-null  str    
 9   add_water        50566 non-null  str    
 10  nutrients_adder  50570 non-null  str    
 11  humidifier       50570 non-null  str    
 12  ex_fan           50570 non-null  str    
dtypes: float64(5), int64(2), str(6)
memory usage: 5.0 MB


In [43]:
df.describe(include='number')

,id,pH,TDS,water_level,DHT_temp,DHT_humidity,water_temp
count,50570.000000,50570.000000,50570.000000,50570.000000,50570.000000,50570.000000,50570.000000
mean,25285.500000,6.001514,1154.134377,1.258829,24.316781,71.714188,21.494904
std,14598.445893,0.553016,247.507196,0.479218,1.209319,25.956493,2.174898
min,1.000000,0.270000,-283.910000,0.000000,12.300000,25.000000,0.000000
25%,12643.250000,5.630000,1020.040000,1.000000,23.600000,67.500000,19.770000
50%,25285.500000,5.790000,1234.650000,1.000000,24.400000,71.700000,21.520000
75%,37927.750000,6.680000,1329.067500,2.000000,25.100000,77.500000,23.277500
max,50570.000000,11.570000,2278.350000,3.000000,70.000000,3312.600000,25.000000


#### Exploratory Data Prep and Cleaning

In [44]:
# fill the missing values with the most frequent state(mode)

df['add_water']=df['add_water'].fillna(df['add_water'].mode()[0])

In [45]:
# drop redundant ID column 

df = df.drop(columns=['id'])

In [46]:
df.columns

Index(['timestamp', 'pH', 'TDS', 'water_level', 'DHT_temp', 'DHT_humidity',
       'water_temp', 'pH_reducer', 'add_water', 'nutrients_adder',
       'humidifier', 'ex_fan'],
      dtype='str')

In [47]:
# handle timestamps 

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

In [48]:
# encode categorical columns to binary outputs 

categorical_columns = df.select_dtypes(include='object').columns

/var/folders/qh/qwbycz4d59792_lr1v_pflyw0000gn/T/ipykernel_71885/2531870578.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include='object').columns


In [49]:
categorical_columns = categorical_columns.tolist()
print(categorical_columns)

['pH_reducer', 'add_water', 'nutrients_adder', 'humidifier', 'ex_fan']


In [50]:
for col in categorical_columns:
  df[col] = df[col].apply(lambda x: 1 if x == "ON" else 0)

##### Feature Engineering

In [51]:
# extract the temporal features from the timestamp 

df['hour'] = df['timestamp'].dt.hour 
df['minute'] = df['timestamp'].dt.minute

In [54]:
# define feature X and multi-output target y 

X = df[['pH', 'TDS', 'water_level', 'DHT_temp', 'DHT_humidity', 'water_temp', 'hour', 'minute']]

y = df[categorical_columns]

In [57]:
X

,pH,TDS,water_level,DHT_temp,DHT_humidity,water_temp,hour,minute
0,7.00,500.00,0,25.5,60.0,20.00,10,57
1,7.00,500.00,0,25.5,60.0,20.00,10,58
2,7.00,500.00,3,25.5,60.0,20.00,11,1
3,7.00,500.00,3,25.5,60.0,20.00,11,30
4,7.00,500.00,3,25.5,60.0,20.00,11,33
...,...,...,...,...,...,...,...,...
50565,5.54,1125.10,1,23.9,77.7,21.82,21,34
50566,5.55,1125.10,1,23.9,77.7,19.83,21,34
50567,5.57,1125.21,1,23.9,77.7,19.41,21,35
50568,5.56,1124.89,1,23.9,77.7,19.88,21,35


In [58]:
y

,pH_reducer,add_water,nutrients_adder,humidifier,ex_fan
0,1,0,0,0,1
1,1,0,0,0,0
2,1,0,0,0,0
3,1,0,0,0,0
4,1,1,0,0,0
...,...,...,...,...,...
50565,0,0,0,0,0
50566,0,0,0,0,0
50567,0,0,0,0,0
50568,0,0,0,0,0


##### Pipeline and Algorithms Setup

In [60]:
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline 
from sklearn.multioutput import MultiOutputClassifier 
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC 
import time

preprocessor = StandardScaler()

In [61]:
models = {
	"RandomForest": MultiOutputClassifier(
		RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
	),
	"HistGradientBoosting": MultiOutputClassifier(
		HistGradientBoostingClassifier(random_state=42)
	),
	"SVM(RBF kernel)": MultiOutputClassifier(
		SVC(kernel='rbf', random_state=42)
	)
}

##### Cross-validation and Evaluation

In [62]:
results = []
print("Starting 5-fold cross-validation")
print("-"*50)

for name, model in models.items():
  print(f"Evaluating {name}.")
  
  # create the pipeline 	
  pipeline = Pipeline(steps=[
		('preprocessor', preprocessor),
		('classifier', model)
	])
  
  start_time = time.time()
  # Execute 5-Fold Cross Validation. 
  # 'accuracy' here evaluates EXACT subset matches (all 5 targets correct)
  cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy', n_jobs=-1)
    
  end_time = time.time()
    
  results.append({
      'Model': name,
      'Mean Accuracy (%)': np.round(np.mean(cv_scores) * 100, 2),
      'CV Std Dev (%)': np.round(np.std(cv_scores) * 100, 4),
      'Time Taken (s)': np.round(end_time - start_time, 2)
    })

results_df = pd.DataFrame(results)
print("\n--- Cross Validation Results ---")
print(results_df.to_string(index=False))

Starting 5-fold cross-validation
--------------------------------------------------
Evaluating RandomForest.
Evaluating HistGradientBoosting.
Evaluating SVM(RBF kernel).

--- Cross Validation Results ---
               Model  Mean Accuracy (%)  CV Std Dev (%)  Time Taken (s)
        RandomForest              89.65         12.9174           11.99
HistGradientBoosting              83.40         20.3944            6.89
     SVM(RBF kernel)              88.70         15.3806           34.39


In [63]:
import joblib 

best_model = models['RandomForest']

# create the final pipeline 
final_pipeline = Pipeline(steps=[
	('preprocessor', preprocessor),
	('classifier', best_model)
])

In [64]:
final_pipeline.fit(X,y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.",list,"[array([0, 1]), array([0, 1]), array([0, 1]), array([0, 1]), ...]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['pH','TDS','water_level',...,'water_temp','hour','minute']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
